# Fine-tune Google Gemma-3 4B with DeepSpeed ZeRO-3 on Amazon SageMaker AI using ModelTrainer

In this notebook, we fine-tune [Google Gemma-3 4B Instruct](https://huggingface.co/google/gemma-3-4b-it) on Amazon SageMaker AI, using Python scripts and SageMaker ModelTrainer for executing a training job with DeepSpeed ZeRO-3 distributed training strategy.

## Overview

- **Model**: google/gemma-3-4b-it
- **Strategy**: DeepSpeed ZeRO-3 with CPU offloading
- **Dataset**: HuggingFaceH4/Multilingual-Thinking (Apache 2.0 license)
- **Training**: LoRA fine-tuning with merged weights

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

## Setup Configuration

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

In [ ]:
sagemaker_session = Session()
sagemaker_session_bucket = sagemaker_session.default_bucket()
sagemaker_session = Session(default_bucket=sagemaker_session_bucket)

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sagemaker_session.boto_region_name}")

Configure your Hugging Face token and optionally MLflow tracking server ARN.

In [ ]:
import os

model_id = "google/gemma-3-4b-it"

os.environ["HF_TOKEN"] = "<HF_TOKEN>"
os.environ["model_id"] = model_id
os.environ["mlflow_uri"] = ""
os.environ["mlflow_experiment_name"] = "gemma-3-4b-it-reasoning-multi-language"

## Visualize and upload the dataset

We are going to load [HuggingFaceH4/Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset (Apache 2.0 license).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.1, random_state=42)
train, test = train_test_split(train, test_size=10, random_state=42)

print("Number of train elements: ", len(train))
print("Number of val elements: ", len(val))
print("Number of test elements: ", len(test))

Create a prompt template and format the dataset using Gemma-3 chat template.

In [ ]:
import textwrap
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def prepare_dataset(sample):
    messages = []
    system_prompt = ""
    first_user_message = True
    
    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """
            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
        elif el["role"] == "user":
            if first_user_message:
                first_user_message = False
                content = (system_prompt + "\n\n" + el["content"]) if system_prompt else el["content"]
                messages.append({"role": "user", "content": content})
            else:
                messages.append({"role": "user", "content": el["content"]})
        else:
            if el["thinking"] is not None and el["thinking"] != "" and el["thinking"] != "null":
                messages.append({
                    "role": "assistant",
                    "content": f"<think>\n{el['thinking']}\n</think>\n{el['content']}",
                })
            else:
                messages.append({"role": "assistant", "content": el["content"]})

    sample["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return sample

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=list(train_dataset.features)
)

print(train_dataset[randint(0, len(train_dataset) - 1)]["text"])

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=list(val_dataset.features)
)

### Upload to Amazon S3

In [ ]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client('s3')

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/gemma-3-4b-it-fine-tuning-dsz3"
else:
    input_path = f"datasets/gemma-3-4b-it-fine-tuning-dsz3"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.json"

In [ ]:
import os

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/val", exist_ok=True)

# Save datasets to s3
# We will fine tune only with 20 records due to limited compute resource for the workshop
train_dataset.to_json("./data/train/dataset.json", orient="records")
val_dataset.to_json("./data/val/dataset.json", orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json")
s3_client.upload_file("./data/val/dataset.json", bucket_name, f"{input_path}/val/dataset.json")

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)

## Model fine-tuning

We are now ready to fine-tune our model. We will use the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) from transformers to fine-tune our model. We prepared a script [train.py](./scripts/train.py) which loads the dataset from disk, prepares the model, tokenizer and starts the training.

### Training configurations

For configuration we use `TrlParser`, that allows us to provide hyperparameters in a `yaml` file. This yaml will be uploaded and provided to Amazon SageMaker similar to our datasets.

In [ ]:
%%bash

cat > ./args.yaml <<EOF
model_id: "${model_id}"                           # Hugging Face model id
mlflow_uri: "${mlflow_uri}"                       # MLflow tracking server URI
mlflow_experiment_name: "${mlflow_experiment_name}" # MLflow experiment name
# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
checkpoint_dir: "/opt/ml/checkpoints/"            # directory for saving training checkpoints
train_dataset_path: "/opt/ml/input/data/train/"   # path to where S3 saves train dataset
val_dataset_path: "/opt/ml/input/data/val/"       # path to where S3 saves test dataset
token: "${HF_TOKEN}"                              # Hugging Face API token
merge_weights: true                               # merge weights in the base model
# training parameters
apply_truncation: true                           # apply truncation to datasets
attn_implementation: "flash_attention_2"         # attention implementation type
learning_rate: 2e-5                              # learning rate scheduler
num_train_epochs: 10                             # number of training epochs
per_device_train_batch_size: 1                   # batch size per device during training
per_device_eval_batch_size: 2                    # batch size for evaluation
eval_strategy: "steps"                           # run evaluation every eval_steps
eval_steps: 100                                  # evaluate every N steps
gradient_accumulation_steps: 16                  # number of steps before performing a backward/update pass
gradient_checkpointing: true                     # use gradient checkpointing
torch_dtype: "bfloat16"                          # float precision type
bf16: true                                       # use bfloat16 precision
tf32: true                                       # use tf32 precision
ignore_data_skip: true                           # skip data loading errors
logging_strategy: "steps"                        # logging strategy
logging_steps: 1                                 # log every N steps
log_on_each_node: false                          # disable logging on each node
ddp_find_unused_parameters: false                # DDP unused parameter detection
save_total_limit: 1                              # maximum number of checkpoints to keep
save_steps: 100                                  # Save checkpoint every this many steps
warmup_steps: 50                                 # number of warmup steps
weight_decay: 0.01                               # weight decay coefficient
dataloader_pin_memory: false                     # pin memory for dataloader
# LoRA parameters
load_in_4bit: false                              # enable 4-bit quantization
lora_r: 16                                       # LoRA rank
lora_alpha: 32                                   # LoRA alpha parameter
lora_dropout: 0.1                                # LoRA dropout rate
EOF

In [ ]:
import os

if default_prefix:
    input_path = f"{default_prefix}/datasets/gemma-3-4b-it-fine-tuning-dsz3"
else:
    input_path = f"datasets/gemma-3-4b-it-fine-tuning-dsz3"

train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

# upload the model yaml file to s3
model_yaml = "args.yaml"
s3_client.upload_file(model_yaml, bucket_name, f"{input_path}/config/args.yaml")
os.remove("./args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

### DeepSpeed ZeRO-3 configurations

In [ ]:
%%bash

cat > ./accelerate_config.yaml <<EOF
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  deepspeed_multinode_launcher: standard
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
distributed_type: DEEPSPEED
downcast_bf16: 'no'
main_training_function: main
mixed_precision: bf16
rdzv_backend: c10d                        # static for single node, c10d for single and multi-node
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
EOF

In [ ]:
import os

if default_prefix:
    input_path = f"{default_prefix}/datasets/gemma-3-4b-it-fine-tuning-dsz3"
else:
    input_path = f"datasets/gemma-3-4b-it-fine-tuning-dsz3"

train_accelerate_config_s3_path = f"s3://{bucket_name}/{input_path}/accelerate_config/accelerate_config.yaml"

# upload the model yaml file to s3
model_yaml = "accelerate_config.yaml"
s3_client.upload_file(model_yaml, bucket_name, f"{input_path}/accelerate_config/accelerate_config.yaml")
os.remove("./accelerate_config.yaml")

print(f"Accelerate config uploaded to:")
print(train_accelerate_config_s3_path)

## Fine-tune model

Below estimator will train the model with LoRA, merge the adapter in the base model and save in S3.

### Get PyTorch image_uri

We are going to use the native PyTorch container image, pre-built for Amazon SageMaker.

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
# ml.g5.12xlarge has 4x A10G GPUs
instance_type = "ml.g5.12xlarge"
instance_count = 1

instance_type

In [ ]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "--entrypoint",
    "train.py",
    "--accelerate_config",
    "/opt/ml/input/data/accelerate_config/accelerate_config.yaml",
    "--config",
    "/opt/ml/input/data/config/args.yaml",
]

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"bash sm_accelerate_train.sh {' '.join(args)}",
)

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=0,
)

job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-dsz3"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
)

In [ ]:
from sagemaker.train.configs import InputData

train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path,
)

val_input = InputData(
    channel_name="val",
    data_source=val_dataset_s3_path,
)

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,
)

accelerate_config_input = InputData(
    channel_name="accelerate_config",
    data_source=train_accelerate_config_s3_path,
)

data = [train_input, val_input, config_input, accelerate_config_input]
data

In [ ]:
# Start the training job
model_trainer.train(input_data_config=data, wait=False)

---

# Model Deployment

In the following sections, we are going to deploy the fine-tuned model on an Amazon SageMaker Real-time endpoint.

In [ ]:
model_id = "google/gemma-3-4b-it"

model_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3"
endpoint_config_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-conf"
endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-endpoint"
ic_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-ic"

## Load Fine-Tuned model

In [ ]:
job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-dsz3"

In [ ]:
import boto3

def get_last_job_name(job_name_prefix):
    sagemaker_client = boto3.client('sagemaker')

    matching_jobs = []
    next_token = None

    while True:
        # Prepare the search parameters
        search_params = {
            'Resource': 'TrainingJob',
            'SearchExpression': {
                'Filters': [
                    {
                        'Name': 'TrainingJobName',
                        'Operator': 'Contains',
                        'Value': job_name_prefix
                    },
                    {
                        'Name': 'TrainingJobStatus',
                        'Operator': 'Equals',
                        'Value': "Completed"
                    }
                ]
            },
            'SortBy': 'CreationTime',
            'SortOrder': 'Descending',
            'MaxResults': 100
        }

        # Add NextToken if we have one
        if next_token:
            search_params['NextToken'] = next_token

        # Make the search request
        search_response = sagemaker_client.search(**search_params)

        # Filter and add matching jobs
        matching_jobs.extend([
            job['TrainingJob']['TrainingJobName'] 
            for job in search_response['Results']
            if job['TrainingJob']['TrainingJobName'].startswith(job_name_prefix)
        ])

        # Check if we have more results to fetch
        next_token = search_response.get('NextToken')
        if not next_token or matching_jobs:  # Stop if we found at least one match or no more results
            break

    if not matching_jobs:
        raise ValueError(f"No completed training jobs found starting with prefix '{job_name_prefix}'")

    return matching_jobs[0]

In [ ]:
job_name = get_last_job_name(job_prefix)

job_name

### Create Endpoint Configuration

Define inference configuration

In [ ]:
instance_count = 1
instance_type = "ml.g5.12xlarge"
number_of_gpu = 4
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
            inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1",
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

### Create Endpoint

A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

### Create Model

Get the image URI

In [ ]:
region = sagemaker_session.boto_region_name
CONTAINER_VERSION = "vllm:0.22.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/{CONTAINER_VERSION}"

image_uri

Create the Gemma 3 jinja chat template

In [ ]:
%%bash

cat > ./tool_chat_template_gemma3_pythonic.jinja <<'EOF'
{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '\n\n' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '\n\n' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}
{%- for message in loop_messages | rejectattr("role", "equalto", "tool") | selectattr("tool_calls", "undefined") -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
{%- endfor -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- elif (message['role'] == 'tool') -%}
        {%- set role = "user" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '\n' -}}
    {%- if loop.first -%}
        {{ first_user_prefix }}
        {%- if tools is not none -%}
            {{- "You have access to the following tools to help respond to the user. To call tools, please respond with a python list of the calls. DO NOT USE MARKDOWN SYNTAX.\n" }}
            {{- 'Respond in the format [func_name1(params_name1=params_value1, params_name2=params_value2...), func_name2(params)] \n' }}
            {{- "Do not use variables.\n\n" }}
            {%- for t in tools -%}
                {{- t | tojson(indent=4) }}
                {{- "\n\n" }}
            {%- endfor -%}
        {%- endif -%}
    {%- endif -%}

    {%- if 'tool_calls' in message -%}
        {{- '[' -}}
        {%- for tool_call in message.tool_calls -%}
            {%- if tool_call.function is defined -%}
                {%- set tool_call = tool_call.function -%}
            {%- endif -%}
            {{- tool_call.name + '(' -}}
            {%- for param in tool_call.arguments -%}
                {{- param + '=' -}}
                {{- "%sr" | format(tool_call.arguments[param]) -}}
                {%- if not loop.last -%}, {% endif -%}
            {%- endfor -%}
            {{- ')' -}}
            {%- if not loop.last -%},{%- endif -%}
        {%- endfor -%}
        {{- ']' -}}
    {%- endif -%}

    {%- if (message['role'] == 'tool') -%}
        {{ '<tool_response>\n' -}}
    {%- endif -%}
    {%- if message['content'] is string -%}
        {{ message['content'] | trim }}
    {%- elif message['content'] is iterable -%}
        {%- for item in message['content'] -%}
            {%- if item['type'] == 'image' -%}
                {{ '<start_of_image>' }}
            {%- elif item['type'] == 'text' -%}
                {{ item['text'] | trim }}
            {%- endif -%}
        {%- endfor -%}
    {%- else -%}
        {{ raise_exception("Invalid content type") }}
    {%- endif -%}
    {%- if (message['role'] == 'tool') -%}
        {{ '</tool_response>' -}}
    {%- endif -%}
    {{ '<end_of_turn>\n' }}
{%- endfor -%}
{%- if add_generation_prompt -%}
    {{'<start_of_turn>model\n'}}
{%- endif -%}
EOF

In [ ]:
import os

input_s3_prefix = f"{job_prefix}/{job_name}/output/model"

# upload the model yaml file to s3
vllm_yaml = "tool_chat_template_gemma3_pythonic.jinja"
s3_client.upload_file(vllm_yaml, bucket_name, f"{input_s3_prefix}/tool_chat_template_gemma3_pythonic.jinja")
os.remove("./tool_chat_template_gemma3_pythonic.jinja")

print(f"Training config uploaded to:")
print(f"s3://{bucket_name}/{input_s3_prefix}/tool_chat_template_gemma3_pythonic.jinja")

Create vLLM configurations for tool parser

In [ ]:
%%bash

cat > ./vllm_config.yaml <<EOF
enable_auto_tool_choice: true
tool_call_parser: pythonic
chat_template: /opt/ml/model/tool_chat_template_gemma3_pythonic.jinja
EOF

In [ ]:
import os

input_s3_prefix = f"{job_prefix}/{job_name}/output/model"

# upload the model yaml file to s3
vllm_yaml = "vllm_config.yaml"
s3_client.upload_file(vllm_yaml, bucket_name, f"{input_s3_prefix}/vllm_config.yaml")
os.remove("./vllm_config.yaml")

print(f"Training config uploaded to:")
print(f"s3://{bucket_name}/{input_s3_prefix}/vllm_config.yaml")

Define engine arguments as environment variables

In [ ]:
import json

env = {
    "SM_VLLM_MODEL": "/opt/ml/model",
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "SM_VLLM_CONFIG": "/opt/ml/model/vllm_config.yaml",
    "SM_VLLM_DTYPE": "bfloat16",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.8",
    "SM_VLLM_MAX_MODEL_LEN": json.dumps(1024 * 32),
    "SM_VLLM_MAX_NUM_SEQS": "32",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)
from rich.pretty import pprint

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=f"s3://{bucket_name}/{job_prefix}/{job_name}/output/model/",
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(fine_tuned_model)

### Create Inference Component

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

# Step 3: Create InferenceComponent
inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=model_name,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=10240,
            number_of_accelerator_devices_required=number_of_gpu,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
inference_component.wait_for_status("InService")
print(f"Endpoint {ic_name} is InService")

***

### Test predict

In [ ]:
import boto3
import io
import json

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

### Iterator class for streaming inference

Utility class to parse streaming responses

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return (content, tool_call_delta)."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None, None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                delta = choice.get("delta", {})
                if "content" in delta and delta["content"]:
                    return delta["content"], None
                if "tool_calls" in delta:
                    return None, delta["tool_calls"]
    except json.JSONDecodeError:
        pass

    return None, None

In [ ]:
system_prompt = f"""
You are an AI assistant that thinks in {{language}} but responds in English.

IMPORTANT: Follow this exact format for every response:
1. First, write your reasoning and thoughts inside <think>...</think> tags
2. Then, provide your final answer in English

Always think through the problem in {{language}}, then translate your conclusion to English for the final response.
"""

prompt = "Hello. Who are you? Please explain"

In [ ]:
system = system_prompt.format(language="Italian")

request_body = {
    "messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""
tool_calls = []

for line in LineIterator(response["Body"]):
    if line:
        content, tool_call_delta = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)
        if tool_call_delta:
            for tc in tool_call_delta:
                idx = tc.get("index", 0)
                while len(tool_calls) <= idx:
                    tool_calls.append(
                        {"type": "function", "function": {"name": "", "arguments": ""}}
                    )
                if "function" in tc:
                    if tc["function"].get("name"):
                        tool_calls[idx]["function"]["name"] = tc["function"]["name"]
                    if "arguments" in tc["function"]:
                        tool_calls[idx]["function"]["arguments"] += tc["function"][
                            "arguments"
                        ]

if tool_calls:
    pprint(tool_calls)

***

### Delete resources

In [ ]:
model_id = "google/gemma-3-4b-it"

model_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3"
endpoint_config_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-conf"
endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-endpoint"
ic_name = f"{model_id.split('/')[-1].replace('.', '-')}-sft-dsz3-ic"

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint (optional - if you want to remove the endpoint too)
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config (optional)
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()